### 01_market_prices_autoloader

In [0]:
%sql
DROP Table vattenfall_dev.raw.bronze_market_prices;

In [0]:
import yaml
from pyspark.sql import functions as F

config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog = config["catalog"]
raw_schema = config["schemas"]["raw"]

landing_volume = config["volumes"]["landing"]
checkpoint_volume = config["volumes"]["checkpoints"]

domain = "market_prices"

In [0]:
landing_path = f"/Volumes/{catalog}/{raw_schema}/{landing_volume}/{domain}/"
checkpoint_path = f"/Volumes/{catalog}/{raw_schema}/{checkpoint_volume}/{domain}/"
schema_path =f"{checkpoint_path}/_schema"

target_table = f"{catalog}.{raw_schema}.bronze_{domain}"

print("Landing:", landing_path)
print("Checkpoint:", checkpoint_path)
print("Table:", target_table)
print("schema path:",schema_path)

In [0]:
# stop any running streams
for s in spark.streams.active:
    s.stop()

# clean old schema + checkpoint
dbutils.fs.rm(checkpoint_path, True)
dbutils.fs.mkdirs(checkpoint_path)

In [0]:
repo_path = f"file:/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/sample_data/{domain}"

dbutils.fs.mkdirs(landing_path)

files = dbutils.fs.ls(repo_path)

for f in files:
    dbutils.fs.cp(f.path, landing_path + f.name)

print (f.path, landing_path + f.name)
display(dbutils.fs.ls(landing_path))

In [0]:

bronze_stream_df = (
 spark.readStream.format("cloudFiles")
 .option("cloudFiles.format", "csv")
 .option("header", "true")
 .option("inferSchema", "true")
 .option("cloudFiles.schemaLocation", schema_path)
 .load(landing_path)
)

In [0]:
query = (
    bronze_stream_df
    .withColumn("ingestion_ts", F.current_timestamp())
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()
print(query.status)
print(query.lastProgress)
display(spark.table(target_table))


In [0]:
%sql
SELECT * FROM vattenfall_dev.raw.bronze_market_prices;